## Regression Modeling using PySpark ML Pipeline

### References:
- https://spark.apache.org/docs/latest/api/python/reference/pyspark.ml.html
- https://kb.databricks.com/machine-learning/extract-feature-info.html

In [ ]:
import os

import math
import matplotlib.pyplot as plt
import numpy as np
import pyspark.sql.functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import MinMaxScaler, VectorAssembler

import geofunctions as S

In [ ]:
schema = ",".join(
    [
        "id string",
        "vendor_id string",
        "pickup_datetime timestamp",
        "dropoff_datetime timestamp",
        "passenger_count integer",
        "pickup_longitude double",
        "pickup_latitude double",
        "dropoff_longitude double",
        "dropoff_latitude double",
        "store_and_fwd_flag string",
        "trip_duration integer",  # seconds
    ]
)

In [ ]:
xmin, xmax, ymin, ymax = [
    -74.04226680414989,
    -73.5061146385722,
    40.6059735081628,
    40.91340235669589,
]

where = " and ".join(
    [
        f"p_lon between {xmin}D and {xmax}D",
        f"p_lat between {ymin}D and {ymax}D",
        f"d_lon between {xmin}D and {xmax}D",
        f"d_lat between {ymin}D and {ymax}D",
        "label between 60 and 5400",
        "haversine between 10D and 32000D"
    ]
)

to_pi = 2.0 * math.pi / 23.0

In [ ]:
csv_path = os.path.expanduser("~/JHUBigDataClass/data/nyc-taxi-trip-duration/train.csv")


df = (
    spark.read.csv(csv_path, schema=schema, sep=",", mode="DROPMALFORMED")
    .drop("id", "vendor_id", "passenger_count", "store_and_fwd_flag")
    .withColumnRenamed("pickup_datetime", "p_date")
    .withColumnRenamed("pickup_longitude", "p_lon")
    .withColumnRenamed("pickup_latitude", "p_lat")
    .withColumnRenamed("dropoff_datetime", "d_date")
    .withColumnRenamed("dropoff_longitude", "d_lon")
    .withColumnRenamed("dropoff_latitude", "d_lat")
    .withColumnRenamed("trip_duration", "label")
    .withColumn("haversine", S.st_haversine("p_lat", "p_lon", "d_lat", "d_lon"))
    .where(where)
    .withColumn("pi", F.hour("p_date") * to_pi)
    .withColumn("hh_cos", F.cos("pi"))
    .withColumn("hh_sin", F.sin("pi"))
    # .withColumn("haversine", F.log("haversine"))
    # .withColumn("label", F.log("label"))
    .drop("pi")
)

In [ ]:
train, valid = df.randomSplit([0.7, 0.3], seed=42)
train = train.cache()

In [ ]:
from pyspark.ml.regression import RandomForestRegressor

reg = RandomForestRegressor(numTrees=64, maxDepth=5, maxBins=32)

In [ ]:
assembler = VectorAssembler(
    inputCols=["p_lon", "p_lat", "d_lon", "d_lat", "haversine", "hh_cos", "hh_sin"],
    # inputCols=["p_lon", "p_lat", "d_lon", "d_lat", "haversine"],
    outputCol="out",
)

scaler = MinMaxScaler(inputCol="out", outputCol="features")

pipeline = Pipeline(stages=[assembler, scaler, reg])

In [ ]:
model = pipeline.fit(train)

In [ ]:
reg_model = model.stages[-1]

In [ ]:
features = np.array(assembler.getInputCols())
importances = reg_model.featureImportances.toArray()
indices = np.argsort(importances)

In [ ]:
plt.title("Feature Importances")
plt.barh(range(len(indices)), importances[indices], color="#8f63f4", align="center")
plt.yticks(range(len(indices)), features[indices])
plt.xlabel("Relative Importance")
plt.show()

In [ ]:
# for feature, importance in zip(assembler.getInputCols(), reg_model.featureImportances):
#     print(f"{feature}: {importance:.4f}")

In [ ]:
predictions = model.transform(valid)

In [ ]:
predictions.select("label", "prediction").show()

In [ ]:
(
    predictions
    .select((F.col("label") - F.col("prediction")).alias("dd"))
    .select((F.col("dd") * F.col("dd")).alias("dd"))
    .agg(F.avg("dd").alias("mse"))
    # .select((F.sqrt("dd") / 60.0).alias("err in min"))
    .show()
)

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction")

In [ ]:
mse = evaluator.evaluate(predictions, {evaluator.metricName: "mse"})
r2 = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})

mse, r2